# Examine Relationship Between Combine Features And DB Separation During Hitch/In/Out Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We've found previously that there are Combine features created from our Combine player-tracking data that are related to in-game performance for WRs. More specifically, WRs with sharp deceleration ability are able to create more separation in cutting routes (Hitch, In, Out routes).

We want to examine if similar relationships hold for DBs who are guarding the WRs. More specifically, are there Combine metrics that we can extract from the Combine player-tracking data that are related to a DBs ability to minimize separation in cutting routes (Hitch, In, Out routes). 

We hypothesize that, similar to how WRs with strong deceleration ability are able to create separation, DBs who have strong acceleration and deceleration ability will be able to minimize the seperation that their WR creates when they cut.

Predictors: Combine Performance Metrics

Outcome: DBs Minimizing Separation At Cut During Hitch/In/Out Routes

**<span style="color: gold;">FINDINGS</span>**

There is a significant relationship between *best_w_decel* (which measures braking efficiency, higher value is better) and *avg_separation_loss_delta* (distance from WR 0.5 seconds after cut - distance from WR at cut, i.e. how much distance they gained towards the WR). Therefore, DBs who show that they can hit the brakes very efficiently have a signficiant association with their ability to minimize the separation the WR creates by a cut.
* p = 0.005, R-squared = 0.75, N = 8.
* *when controlling for general agility (from accel_score)*: p = 0.009, R-squared = 0.85, N = 7

No other metrics (acceleration, deceleration, or cutting metrics) have a meaningful relationship with the DBs in-game performance with respect to minimizing separation.

**<span style="color: gold;">NOTES</span>**

Since we need to create outcome variables related to separation and since we need to know who is guarding who, we must use player-tracking data from the 2026 Big Data Bowl. That data only includes 2023 Regular season games. Therefore, our analysis will only include 2023 rookies during 2023 Regular season time.

While we usually only look at man coverage situations, we're too data starved to exclude zone coverage plays. Therefore, this analysis includes both man and zone coverage plays.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["HITCH", "IN", "OUT"]

<br><br/>
#### DB Separation Data From *11_BDB26_DB_Player_Tracking.ipynb*

In [ ]:
# =========================================
# DB Separation Data Among 2023 Rookies
# =========================================

db_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "11_2023_Rookie_DB_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    db_separation_df
    .select("defender_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

# ----------------------------------------
# Only Look At Specified Routes
# ----------------------------------------

db_separation_df = (
    db_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)

# ----------------------------------------
# Only Look At Man Coverage Situations
# ----------------------------------------

# db_separation_df = (
#     db_separation_df
#     .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
# )

print("Separation Data:")
print(f"    - Dimensions: {db_separation_df.shape}")
print(f"    - Number of Rookie DBs: {db_separation_df.select("defender_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {db_separation_df.select("player_play_id").n_unique()}")
db_separation_df

In [ ]:
# =======================================
# Num Player-Plays Per Player
# =======================================

(
    db_separation_df
    .group_by("defender_nfl_id")
    .agg(num_player_plays = pl.col("player_play_id").n_unique())
    .sort("num_player_plays", descending = True)
)

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Features Data
# =======================================

combine_features_df = pl.read_csv(str(SEPARATION_DATA_DIR / "13_DB_Combine_Features_Data.csv"))

print("Combine Features Data:")
print(f"    - Dimensions: {combine_features_df.shape}")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

<br><br/>
## Data Cleaning
***

#### Get Data On Cuts And Separation At Cut From Targeted Hitch/In Routes

In [ ]:
# ==============================================================================
# Extract DB Closeness at the WR's Cut Apex for Hitch/In Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate WR Kinematics (Speed, Acceleration, and Downfield Velocity)
# ------------------------------------------------------------------------------
kinematics_df = (
    db_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities for the WR (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # WR Speed is the magnitude of the velocity vector
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # Calculate true linear acceleration of the WR
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate the WR's Cut Apex Frame Per Play
# ------------------------------------------------------------------------------
cut_apexes_df = (
    kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("v_downfield") > 4.0)
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where the WR slammed on the brakes the hardest
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").min()).first().alias("cut_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract DB Closeness and Calculate the Separation Delta
# ------------------------------------------------------------------------------
final_in_game_closeness_df = (
    kinematics_df
    .join(cut_apexes_df, on="player_play_id", how="inner")
    # Grab the metrics specifically at the cut apex or 0.5s right after it
    .filter(
        (pl.col("frame_id") == pl.col("cut_frame_id")) | 
        (pl.col("frame_id") == (pl.col("cut_frame_id") + 5))
    )
    .group_by("player_play_id")
    .agg([
        pl.col("defender_nfl_id").first(),
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        pl.col("route").first(),
        
        # Distance to WR exactly at the moment of the cut plant
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("cut_frame_id")).first().alias("distance_at_apex"),
        # Distance to WR 0.3 seconds after the cut
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("cut_frame_id") + 5)).first().alias("distance_post_cut")
    ])
    .with_columns(
        # Positive values = DB gave up separation. Negative values = DB closed the gap.
        separation_loss_delta = pl.col("distance_post_cut") - pl.col("distance_at_apex")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to ONE row per DB to get their average coverage tightness
# ------------------------------------------------------------------------------
db_closeness_at_cut_df = (
    final_in_game_closeness_df
    .group_by("defender_nfl_id")
    .agg([
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        pl.col("distance_at_apex").mean().alias("avg_distance_at_apex"),
        pl.col("distance_post_cut").mean().alias("avg_distance_post_cut"),
        pl.col("separation_loss_delta").mean().alias("avg_separation_loss_delta")
    ])
)

db_closeness_at_cut_df


<br><br/>
#### Add Combine Metrics To Player Separation Data

In [ ]:
analysis_df = (
    db_closeness_at_cut_df
    .join(
        combine_features_df, 
        left_on="defender_nfl_id", 
        right_on="nfl_id", 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("defender_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

#### Best Decel From W Drill vs Separation Lost

In [ ]:
predictor = "best_w_decel" 
outcome = "avg_separation_loss_delta"

***Scatter Plot***

In [ ]:
fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols", 
    title = "Deceleration Score vs Average Separation Change During Cut", 
    labels = {
        "best_w_decel": "W-Drill Deceleration (yards/second^2)", 
        "avg_separation_loss_delta": "Average Change In Separation (yards)"
    }, 
    template = "simple_white"
)

fig.update_traces(
    line=dict(color="grey"),
    selector=dict(mode="lines")
)

fig.update_traces(
    marker=dict(size=10),
    selector=dict(mode="markers")
)

fig.show()

<br><br/>
***Univariate Model***

In [ ]:
# ===============================================
# Univariate Regression
# ===============================================

# Does the relationship between best_w_decel and avg_separation_loss_delta
# hold if we adjust for general agility?

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

<br><br/>
***Leave-One-Out Analysis***

In [ ]:
# ==========================================
# Leave-One-Out Sensitivity Analysis
# ==========================================

# ----------------------------------------
# Calculate LOO Metrics
# ----------------------------------------

# 1. Prepare regression data
regression_df = (
    analysis_df
    .filter(
        pl.col(predictor).is_not_null()
        & pl.col(outcome).is_not_null()
    )
)

pd_reg = regression_df.to_pandas().reset_index(drop=True)

n = len(pd_reg)

# Check sample size
print(f"Sample Size: {n}")

# 2. Fit a model leaving out each observation
loo_results = []

for i in range(n):

    # Remove one observation
    loo_df = pd_reg.drop(index=i)

    # Define predictor and outcome
    X = sm.add_constant(loo_df[[predictor]])
    y = loo_df[outcome]

    # Fit OLS model
    model = sm.OLS(y, X).fit()

    # Store results for the predictor (excluding intercept)
    loo_results.append({
        "omitted_observation": i,
        "coef": model.params[predictor],
        "p_value": model.pvalues[predictor],
        "r_squared": model.rsquared,
    })

# 3. Create results table
loo_results_df = pd.DataFrame(loo_results)

# ---------------------------------------
# Scatter Plot of P-Values
# ---------------------------------------

fig = px.line(
    data_frame = loo_results_df, 
    x = "omitted_observation", 
    y = "p_value", 
    labels = {
        "p_value": "P-Value", 
        "omitted_observation": "Iteration"
    },
    title = "P-Values From Leave-One-Out Analysis", 
    template = "simple_white"
)

fig.update_yaxes(
    range = [0, 0.1 + 0.01]
)

fig.add_hline(
    y = 0.10, 
    line_dash = "dash", 
    line_color = "red", 
    line_width = 2
)

fig.show()

In [ ]:
# ==========================================
# Leave-One-Out Sensitivity Analysis (cont'd)
# ==========================================

# ------------------------------------------
# Full Model
# ------------------------------------------

X_full = sm.add_constant(pd_reg[[predictor]])
y_full = pd_reg[outcome]

full_model = sm.OLS(y_full, X_full).fit()

full_results = {
    "coef": full_model.params[predictor],
    "p_value": full_model.pvalues[predictor],
    "r_squared": full_model.rsquared,
}

# ------------------------------------------
# Leave-One-Out Summary
# ------------------------------------------

summary_df = pd.DataFrame({
    "Full Model": full_results,
    "LOO Mean": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].mean(),
    "LOO Minimum": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].min(),
    "LOO Maximum": loo_results_df[
        ["coef", "p_value", "r_squared"]
    ].max(),
})

display(summary_df.round(2))

<br><br/>
***Multivariate Model***

In [ ]:
# ===========================================
# Multiple Linear Regression
# ===========================================

# Does the relationship between best_w_decel and avg_separation_loss_delta
# hold if we adjust for general agility (accel_score)?

predictors = [
    "accel_score",
    "best_w_decel", 
]

corr = round(analysis_df.select(pl.corr(predictors[0], predictors[1])).item(), 2)
vif = 1 / (1 - (corr**2))
print(f"Correlation Between {predictors[0]} and {predictors[1]}: {corr} (VIF = {round(vif, 2)})\n")

regression_df = analysis_df.filter(
    pl.all_horizontal(
        [pl.col(col).is_not_null() for col in predictors + [outcome]]
    )
)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictors]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==============================================================================
# 1. Prepare Player-Level Data
# ==============================================================================

predictor = "best_w_decel"
outcome = "avg_separation_loss_delta"
player_id = "defender_nfl_id"

player_df = (
    analysis_df
    .filter(
        pl.col(predictor).is_not_null()
        & pl.col(outcome).is_not_null()
    )
    .select([player_id, predictor, outcome])
    .to_pandas()
)

# Ensure one row per player
assert player_df[player_id].is_unique, (
    "analysis_df must contain one row per WR."
)

eligible_players = player_df[player_id].unique()

print(f"Eligible players: {len(eligible_players)}")


# ==============================================================================
# 2. Baseline Model: Player-Level OLS
# ==============================================================================

baseline_model = smf.ols(
    formula=f"{outcome} ~ {predictor}",
    data=player_df,
).fit()

baseline_coef = baseline_model.params[predictor]
baseline_se = baseline_model.bse[predictor]
baseline_p = baseline_model.pvalues[predictor]
baseline_ci = baseline_model.conf_int().loc[predictor]


# ==============================================================================
# 3. Prepare Play-Level Data for Mixed-Effects Model
# ==============================================================================

play_df = (
    final_in_game_closeness_df
    .filter(
        pl.col(player_id).is_in(eligible_players.tolist())
        & pl.col("separation_loss_delta").is_not_null()
        & pl.col("route").is_in(ROUTES_OF_INTEREST)
    )
    .select([
        "player_play_id",
        player_id,
        "route",
        "separation_loss_delta",
    ])
    .to_pandas()
)

# Confirm one observation per player-play
assert play_df["player_play_id"].is_unique, (
    "Expected one row per player-play."
)

# Attach each player's deceleration score
play_df = play_df.merge(
    player_df[[player_id, predictor]],
    on=player_id,
    how="inner",
    validate="many_to_one",
)

# Remove any remaining missing values
play_df = play_df.dropna(
    subset=[
        "separation_loss_delta",
        predictor,
        "route",
        player_id,
    ]
).copy()

play_df["route"] = play_df["route"].astype("category")

# Check actual sample after filtering
print(f"Players in mixed model: {play_df[player_id].nunique()}")
print(f"Player-plays in mixed model: {len(play_df)}")

print("\nRoute counts:")
print(play_df["route"].value_counts())



# ==============================================================================
# 4. Mixed-Effects Model
#    Fixed effects: best_w_decel + route
#    Random intercept: WR
# ==============================================================================

mixed_model = smf.mixedlm(
    formula="separation_loss_delta ~ best_w_decel + C(route)",
    data=play_df,
    groups=play_df[player_id],
    re_formula="1",
)

mixed_fit = mixed_model.fit(
    reml=False,
    method="lbfgs",
    maxiter=2000,
)

print("\nMixed-Effects Model Summary:")
print(mixed_fit.summary())


# ==============================================================================
# 5. Compare the Deceleration Coefficients
# ==============================================================================

mixed_coef = mixed_fit.fe_params[predictor]
mixed_se = mixed_fit.bse_fe[predictor]
mixed_p = mixed_fit.pvalues[predictor]
mixed_ci = mixed_fit.conf_int().loc[predictor]

comparison_df = pd.DataFrame([
    {
        "model": "Baseline OLS",
        "predictor": predictor,
        "coef": baseline_coef,
        "std_error": baseline_se,
        "p_value": baseline_p,
        "ci_lower": baseline_ci.iloc[0],
        "ci_upper": baseline_ci.iloc[1],
        "n_players": int(baseline_model.nobs),
        "n_plays": int(baseline_model.nobs),
        "r_squared": baseline_model.rsquared,
        "AIC": baseline_model.aic,
    },
    {
        "model": "Route-adjusted mixed effects",
        "predictor": predictor,
        "coef": mixed_coef,
        "std_error": mixed_se,
        "p_value": mixed_p,
        "ci_lower": mixed_ci.iloc[0],
        "ci_upper": mixed_ci.iloc[1],
        "n_players": play_df[player_id].nunique(),
        "n_plays": len(play_df),
        "r_squared": np.nan,
        "AIC": mixed_fit.aic,
    },
])

print("\nDeceleration Effect Comparison:")
print(comparison_df.round(4).to_string(index=False))


# ==============================================================================
# 6. Quantify How Much the Deceleration Coefficient Changed
# ==============================================================================

coef_change = mixed_coef - baseline_coef

if baseline_coef != 0:
    pct_change = 100 * (mixed_coef - baseline_coef) / abs(baseline_coef)
else:
    pct_change = np.nan

print(f"\nBaseline coefficient: {baseline_coef:.4f}")
print(f"Route-adjusted coefficient: {mixed_coef:.4f}")
print(f"Absolute coefficient change: {coef_change:.4f}")
print(f"Change relative to baseline magnitude: {pct_change:.1f}%")
print(f"Mixed model converged: {mixed_fit.converged}")
